# 08 — HBM bandwidth & capacity simulation

**Learning goal.** See exactly how parameters, optimizer state, and activations consume HBM, and how activation memory scales with batch size and sequence length. The chart "HBM utilisation vs (batch, seq_len)" is the canonical OOM diagnostic.

**What you'll observe.**
- Per-category allocation of params, optimizer state, activations, workspace.
- HBM utilisation as a function of `batch_size` (MLP case).
- HBM utilisation as a function of `seq_len` (transformer case — quadratic growth from attention scores).

**Knobs to tune.**
- `tpu_version` — selects HBM capacity per chip.
- `hidden_size`, `num_layers`.
- `dtype` — `bf16` halves memory vs `fp32`.

**Failure modes to watch.**
- `bytes` returned by activation estimators are **per layer** — you must sum.
- Quadratic blow-up at long seq_len: doubling `seq_len` quadruples attention-score memory.
- The simulator silently increments `oom_events` instead of crashing when an allocation overflows.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Allocate params + opt state + activations into HbmSimulator

In [ ]:
from cloud_tpu_lab.src.memory.hbm_sim import make_hbm_for_spec
from cloud_tpu_lab.src.memory.activation_memory import (
    estimate_mlp_activations, estimate_transformer_activations, total_bytes,
)
from cloud_tpu_lab.src.tpu_versions.cloud_tpu_catalog import get_spec

spec = get_spec("v5e")
hbm = make_hbm_for_spec(spec)
print(f"HBM capacity: {hbm.capacity_bytes/1e9:.1f} GB on {spec.code_name}")

# Tiny transformer-ish numbers.
hidden = 1024
n_layers = 8
batch = 8
seq_len = 256

# Params (rough): per-layer Linear weights, 4*hidden*hidden each (Q,K,V,O)
elem = 2  # bf16
params_bytes = n_layers * 4 * hidden * hidden * elem
hbm.allocate("params", "parameters", params_bytes)
hbm.allocate("opt_state", "optimizer", params_bytes * 2)  # Adam m,v

acts = estimate_transformer_activations(batch, seq_len, hidden, n_layers, dtype="bf16")
hbm.allocate("activations", "activations", total_bytes(acts))
hbm.allocate("workspace", "workspace", 64 * 1024 * 1024)

print("used  :", hbm.used_bytes() / 1e9, "GB")
print("free  :", hbm.free_bytes() / 1e9, "GB")
print("util  :", hbm.utilization())
print("by category:")
for k, v in hbm.by_category().items():
    print(f"  {k:<14} {v/1e9:8.3f} GB")

## Plot: HBM utilisation vs batch_size (MLP)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

spec = get_spec("v5e")
hidden = 1024
n_layers = 8
params_bytes = n_layers * 4 * hidden * hidden * 2
opt_bytes = params_bytes * 2

batches = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
utils = []
for b in batches:
    hbm = make_hbm_for_spec(spec)
    hbm.allocate("params", "parameters", params_bytes)
    hbm.allocate("opt_state", "optimizer", opt_bytes)
    acts = estimate_mlp_activations(b, hidden, n_layers, dtype="bf16")
    hbm.allocate("activations", "activations", total_bytes(acts))
    hbm.allocate("workspace", "workspace", 64 * 1024 * 1024)
    utils.append(hbm.utilization())

fig = plt.figure(figsize=(7, 4))
plt.plot(batches, utils, marker="o")
plt.axhline(1.0, color="red", linestyle="--", label="OOM threshold")
plt.axhline(0.85, color="orange", linestyle=":", label="warning (85%)")
plt.xscale("log", base=2)
plt.xlabel("batch_size")
plt.ylabel("HBM utilisation")
plt.title(f"MLP HBM utilisation vs batch — hidden={hidden}, layers={n_layers}, {spec.code_name}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb08_hbm_vs_batch.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Plot: HBM utilisation vs seq_len (transformer)

Activations include `B * S * S` attention scores — quadratic in seq_len.

In [ ]:
spec = get_spec("v5e")
hidden = 1024
n_layers = 8
batch = 4
params_bytes = n_layers * 4 * hidden * hidden * 2
opt_bytes = params_bytes * 2

seqs = [64, 128, 256, 512, 1024, 2048, 4096]
utils = []
for s in seqs:
    hbm = make_hbm_for_spec(spec)
    hbm.allocate("params", "parameters", params_bytes)
    hbm.allocate("opt_state", "optimizer", opt_bytes)
    acts = estimate_transformer_activations(batch, s, hidden, n_layers, dtype="bf16")
    hbm.allocate("activations", "activations", total_bytes(acts))
    hbm.allocate("workspace", "workspace", 64 * 1024 * 1024)
    utils.append(hbm.utilization())

fig = plt.figure(figsize=(7, 4))
plt.plot(seqs, utils, marker="o")
plt.axhline(1.0, color="red", linestyle="--", label="OOM threshold")
plt.axhline(0.85, color="orange", linestyle=":", label="warning (85%)")
plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("seq_len")
plt.ylabel("HBM utilisation (log)")
plt.title(f"Transformer HBM utilisation vs seq — batch={batch}, hidden={hidden}, {spec.code_name}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb08_hbm_vs_seq.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## What dominates?

On most modern transformer configs, **activations dominate** HBM, not parameters. That's why:

- **Mixed precision (bf16)** halves activation bytes — usually safe.
- **Gradient checkpointing** trades compute for memory by recomputing activations on the backward pass.
- **Sequence parallelism / activation sharding** spreads activations across chips so per-chip pressure drops.

## Exercises

1. Repeat the seq_len sweep with `spec = get_spec("v5p")` (96 GB HBM). Where does the OOM threshold move to?
2. Plot, on one figure, params/opt/activations contributions stacked as `seq_len` grows. Which crosses over first?
3. Switch to `dtype="fp32"`. Does the OOM seq_len halve?
4. Implement an "activation checkpointing" approximation: assume only `sqrt(n_layers)` activations live at once. Replot vs seq_len.
5. For a target HBM budget of 80%, what's the max `seq_len` you can run with `batch=4, hidden=2048, layers=24` on v5p?